In [ ]:
import scanpy as sc
import anndata as ad
import scipy

from rpy2.robjects import pandas2ri


import anndata2ri

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

In [ ]:
adata_f = sc.read('./Data/RNA_ADT/neurIPS/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad')
adata_gex = adata_f
adata_adt = sc.AnnData(X = adata_f.obsm['protein_counts'])

In [4]:
%%R
suppressPackageStartupMessages({
    library(SingleCellExperiment)
    library(Seurat)
})


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages


In [5]:
adata_ = ad.AnnData(adata_gex.layers['counts'])
adata_.obs_names = adata_gex.obs_names
adata_.var_names = adata_gex.var_names
adata_.obs['cell_type'] = adata_gex.obs['cell_type']
adata_.obs['batch'] = adata_gex.obs['batch']

In [6]:
%%R -i adata_
rna = as.Seurat(adata_, counts='X', data=NULL)
rna

An object of class Seurat 
4000 features across 90261 samples within 1 assay 
Active assay: originalexp (4000 features, 0 variable features)
 2 layers present: counts, data


In addition: Warning messages:
1: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages
2: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages
3: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages
4: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages


In [7]:
adata_ = ad.AnnData(adata_adt.layers['counts'])
adata_.obs_names = adata_adt.obs_names
adata_.var_names = adata_adt.var_names
adata_.obs['cell_type'] = adata_adt.obs['cell_type']
adata_.obs['batch'] = adata_adt.obs['batch']

In [8]:
%%R -i adata_
cite = as.Seurat(adata_, counts='X', data=NULL)
cite

An object of class Seurat 
134 features across 90261 samples within 1 assay 
Active assay: originalexp (134 features, 0 variable features)
 2 layers present: counts, data


In [9]:
%%R
rna <- RenameAssays(rna, originalexp="RNA")
rna.list <- SplitObject(rna, split.by = "batch")
rna.list <- lapply(X = rna.list, FUN = SCTransform, variable.features.n = 1000)
features <- SelectIntegrationFeatures(object.list = rna.list, nfeatures = 1000)
rna.list <- PrepSCTIntegration(object.list = rna.list, anchor.features = features)

  |                                                  | 0 % ~calculating   |+++++                                             | 8 % ~06s           |+++++++++                                         | 17% ~05s           |+++++++++++++                                     | 25% ~05s           |+++++++++++++++++                                 | 33% ~05s           |+++++++++++++++++++++                             | 42% ~05s           |+++++++++++++++++++++++++                         | 50% ~04s           |++++++++++++++++++++++++++++++                    | 58% ~04s           |++++++++++++++++++++++++++++++++++                | 67% ~04s           |++++++++++++++++++++++++++++++++++++++            | 75% ~03s           |++++++++++++++++++++++++++++++++++++++++++        | 83% ~02s           |++++++++++++++++++++++++++++++++++++++++++++++    | 92% ~01s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=10s  


Renaming default assay from originalexp to RNA
Running SCTransform on assay: RNA
vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.
`vst.flavor` is set to 'v2' but could not find glmGamPoi installed.
Please install the glmGamPoi package for much faster estimation.
--------------------------------------------
install.packages('BiocManager')
BiocManager::install('glmGamPoi')
--------------------------------------------
Falling back to native (slower) implementation.

Calculating cell attributes from input UMI matrix: log_umi
Variance stabilizing transformation of count matrix of size 4000 by 5227
Model formula is y ~ log_umi
Get Negative Binomial regression parameters per gene
Using 2000 genes, 5000 cells
Found 21 outliers - those will be ignored in fitting/regularization step

Second step: Get residuals using fitted parameters for 4000 genes
Computing corrected count matrix for 4000 genes
Calculating gene attributes
Wall clock passed: Time difference of 10.75

In [10]:
%%R
anchors <- FindIntegrationAnchors(object.list = rna.list, normalization.method = "SCT", anchor.features = features)
integrated <- IntegrateData(anchorset = anchors, normalization.method = "SCT")
integrated <- RunPCA(integrated)

  |                                                  | 0 % ~calculating   |+                                                 | 2 % ~01h 42m 10s   |++                                                | 3 % ~01h 49m 22s   |+++                                               | 5 % ~01h 42m 06s   |++++                                              | 6 % ~01h 54m 29s   |++++                                              | 8 % ~01h 57m 37s   |+++++                                             | 9 % ~02h 03m 05s   |++++++                                            | 11% ~01h 56m 19s   |+++++++                                           | 12% ~01h 50m 12s   |+++++++                                           | 14% ~01h 47m 45s   |++++++++                                          | 15% ~01h 55m 52s   |+++++++++                                         | 17% ~01h 54m 28s   |++++++++++                                        | 18% ~01h 52m 59s   |++++++++++                                        | 20% ~01h 

Finding all pairwise anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 10439 anchors
Filtering anchors
	Retained 9438 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 12621 anchors
Filtering anchors
	Retained 11291 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 10756 anchors
Filtering anchors
	Retained 9514 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 15025 anchors
Filtering anchors
	Retained 11684 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 13082 anchors
Filtering anchors
	Retained 10221 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 13132 anchors
Filtering anchors
	Retained 10103 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Found 11440 anchors
Filtering anchors
	Retained 10079 anchors
Running CCA
Merging objects
Finding neighborhoods
Finding anchors
	Fou

In [11]:
%%R
cite <- RenameAssays(cite, originalexp='ADT')

cite.list <- SplitObject(cite, split.by = "batch")

cite.list <- lapply(X = cite.list, FUN = function(x) {
    VariableFeatures(x) <- rownames(x[["ADT"]])
    x <- NormalizeData(x, normalization.method = 'CLR', margin = 2, verbose=FALSE)
})

features <- SelectIntegrationFeatures(object.list = cite.list)

cite.list <- lapply(X = cite.list, FUN = function(x) {
    x <- ScaleData(x, features = features, verbose=FALSE)
    x <- RunPCA(x, features = features, reduction.name = "pca", verbose=FALSE)
})

anchors <- FindIntegrationAnchors(object.list = cite.list, reduction = "rpca", 
    dims = 1:30, verbose=FALSE)
integrated_adt <- IntegrateData(anchorset = anchors, dims = 1:30)

integrated_adt <- ScaleData(integrated_adt, verbose=FALSE)
integrated_adt <- RunPCA(integrated_adt, reduction.name = "apca", verbose=FALSE)

  |                                                  | 0 % ~calculating   |+                                                 | 2 % ~04m 33s       |++                                                | 3 % ~04m 40s       |+++                                               | 5 % ~04m 35s       |++++                                              | 6 % ~04m 58s       |++++                                              | 8 % ~05m 07s       |+++++                                             | 9 % ~05m 17s       |++++++                                            | 11% ~05m 04s       |+++++++                                           | 12% ~04m 52s       |+++++++                                           | 14% ~04m 45s       |++++++++                                          | 15% ~04m 48s       |+++++++++                                         | 17% ~04m 46s       |++++++++++                                        | 18% ~04m 42s       |++++++++++                                        | 20% ~04m 

Renaming default assay from originalexp to ADT
Merging dataset 1 into 3
Extracting anchors for merged samples
Finding integration vectors
Finding integration vector weights
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Integrating data
Merging dataset 11 into 10
Extracting anchors for merged samples
Finding integration vectors
Finding integration vector weights
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Integrating data
Merging dataset 2 into 3 1
Extracting anchors for merged samples
Finding integration vectors
Finding integration vector weights
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Integrating data
Merging dataset 12 into 10 11
Extracting an

In [12]:
%%R
integrated[["IADT"]] <- integrated_adt[["integrated"]]
integrated[["apca"]] <- integrated_adt[["apca"]]

integrated <- FindMultiModalNeighbors(integrated, reduction.list = list("pca", "apca"), 
                              dims.list = list(1:50, 1:30), modality.weight.name = "RNA.weight")

integrated <- RunSPCA(integrated, assay = 'integrated', graph = 'wsnn', npcs = 20)

  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~34s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=01m 06s
  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~07s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=17s  
  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~01m 52s       |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=03m 56s
  |                                                  | 0 % ~calculating   |+++++++++++++++++++++++++                         | 50% ~19s           |++++++++++++++++++++++++++++++++++++++++++++++++++| 100% elapsed=29s  


Calculating cell-specific modality weights
Finding 20 nearest neighbors for each modality.
Calculating kernel bandwidths
Finding multimodal neighbors
Constructing multimodal KNN graph
Constructing multimodal SNN graph
Computing sPCA transformation
In addition: Warning messages:
1: Key ‘integrated_’ taken, using ‘iadt_’ instead 
2: Key ‘PC_’ taken, using ‘apca_’ instead 
3: In FindMultiModalNeighbors(integrated, reduction.list = list("pca",  :
  The number of provided modality.weight.name is not equal to the number of modalities. integrated.weight integrated.weight are used to store the modality weights


In [14]:
%%R -o spca
spca = Embeddings(object = integrated[["spca"]])

In [15]:
adata = sc.AnnData(spca)
adata.obs = adata_.obs
adata

AnnData object with n_obs × n_vars = 90261 × 20
    obs: 'cell_type', 'batch'

In [16]:
%%R -o wnn
wnn <- as.data.frame(summary(integrated@graphs$wknn))

In [ ]:
wnn['i'] = wnn['i'] - 1
wnn['j'] = wnn['j'] - 1
adata.obsp['wnn_connectivities'] = scipy.sparse.coo_matrix((wnn['x'], (wnn['i'], wnn['j'])))
adata.obsp['wnn_connectivities'] = scipy.sparse.csr_matrix(adata.obsp['wnn_connectivities'])
adata.write('Seurat_neurIPS.h5ad')